In [30]:
# exracting checkable factual claims from the retracted paper abstracts
# importing the necessary packages

import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# loading the environmen variables 
load_dotenv()

# setting the API key
api_key = os.getenv('OPENAI_API_KEY')
print('key loaded:', (api_key[:7] + '...') if api_key else 'not found')

# setting up the OpenAI connection
client = OpenAI(api_key = api_key)

key loaded: sk-proj...


In [31]:
# setting up the function to call the LLM - can be used for any LLM

# sending a prompt to the LLM and returning a text response
# temperature set to 0 to make the output deterministic (needed for reproducibility)
def call_llm(prompt, model = 'gpt-4o-mini', temperature = 0):
    response = client.chat.completions.create(
        model = model,
        temperature = temperature,
        messages = [{'role': 'user', 'content': prompt}],)

    return response.choices[0].message.content.strip()

In [32]:
# the promp's two jobs are to 1) extract the paper's main finding as one factual claim
# and 2) judge whether the abstract actually contains a claim
# setting to a JSON reponse for more reliable parsing

EXTRACTION_PROMPT = '''You are helping build a fact-verification benchmark from biomedical and life sciences paper abstracts.

Given the abstract: 
1. Decide whether it states a single, specific, and checkable factual finding (measureable effect, relationship or quantity).
Qualitative/interpretive studies, pure methodology descriptions, or reviews do NOT count.
2. If yes, extract the finding as ONE concise, self-contained declarative claim (one sentence, no citations, understandable without the abstract).

Respond with JSON and copy this format exactly:
{{"checkable": true or false, "claim": "the one-sentence claim or empty string if not checkable"}}
Do not add anything else to the response.

Abstract: {abstract}'''

In [33]:
# necessary for the abstract responses
import json

# designing the function to extract the claims
def extract_claim(abstract):

    # runs one abstract through the LLM to return checkable: bool and claim: str
    # returns False if anything fails
    prompt = EXTRACTION_PROMPT.format(abstract = abstract)
    try: 
        raw = call_llm(prompt)

        # stripping potential JSON wrap formatting before parsing
        raw = raw.replace('```json', '').replace('```', '').strip()
        data = json.loads(raw)
        return bool(data.get('checkable', False)), data.get('claim', '').strip()
    
    # displays any errors that occur
    except Exception as e:
        print('parsing/LLM error:', e)
        return False, ''

In [34]:
# loading the cleaned data pool
pool = pd.read_csv('../data/interim/tier1_pool.csv', dtype = str, keep_default_na = False)
print('full data pool:', len(pool))

# small sample of the papers that will be used to build the checkable pairs
## SAMPLE_SIZE = 20 

# scaling up 
SAMPLE_SIZE = 800
sample = pool.sample(SAMPLE_SIZE, random_state = 42).copy()

print('sample:', len(sample))

full data pool: 14984
sample: 800


In [ ]:
# shortcut: run this instead of the extraction/retrieval/gate to load frozen data from disk
# skip during a clean run 

claims_df = pd.read_csv('../data/interim/claims_stage1.csv', keep_default_na = False)
retrieval_df = pd.read_pickle('../data/interim/retrieval_stage2.pkl')
gate_df = pd.read_csv('../data/interim/gate_stage3.csv', keep_default_na = False)
print('reloaded frozen stages')

In [35]:
# grabbing the abstracts from NCBI
from Bio import Entrez
import time
Entrez.email = 'kere00003@uni-saarland.de'   
Entrez.api_key = None 

# function to get abstracts
def fetch_abstract(pmid): 

    # get the abstract for a PubMed ID, if there is none, try
    try: 
        h = Entrez.efetch(db = 'pubmed', id = str(pmid), rettype = 'abstract', retmode = 'xml')
        rec = Entrez.read(h); h.close()
        art = rec['PubmedArticle'][0]['MedlineCitation']['Article']
        parts = art.get('Abstract', {}).get('AbstractText', [])
        return " ".join(str(p) for p in parts).strip()
    except Exception:
        return ''

# fetcihng an abstract for each paper in the sample pool 
abstracts = []
for p in sample['OriginalPaperPubMedID']:
    abstracts.append(fetch_abstract(p))

    # setting retrieval rate
    time.sleep(0.34)   
sample['abstract'] = abstracts

# dropping papers with no usable abstract
sample = sample[sample['abstract'].str.split().str.len() > 20].copy()

print('papers with usable abstracts:', len(sample))

papers with usable abstracts: 792


In [36]:
# running each sampled abstract through the LLM to extract a checkable claim

# collecting one dictionary per paper
results = []

# looping over each paper in the sample (iterrows gives index, row — we ignore the index with _)
for i, (_, row) in enumerate(sample.iterrows()):

    # runs the abstract through extract_claim and returns (checkable: bool, claim: str)
    checkable, claim = extract_claim(row['abstract'])

    # storing the given paper's information + the extraction result
    results.append({

        # allows us to trace back to the original paper
        'pmid': row['OriginalPaperPubMedID'],

        'title': row['Title'],

        # retraction reason for later analysis
        'reason': row['Reason'],
        'abstract': row['abstract'],

        # did the LLM find a checkable claim?
        'checkable': checkable,

        # returns the extracted claim or an empty result
        'claim': claim,
    })
    # setting a pause between API calls to control the rate 
    time.sleep(0.2)

    # progress indicator
    if (i + 1) % 50 == 0:
        print(f'  extracted {i + 1} / {len(sample)}')

# turning the list of dicts into a dataframe for easier inspection
claims_df = pd.DataFrame(results)

# printing the results
print('total processed:', len(claims_df))
print('checkable claims:', claims_df['checkable'].sum())

  extracted 50 / 792
  extracted 100 / 792
  extracted 150 / 792
  extracted 200 / 792
  extracted 250 / 792
  extracted 300 / 792
  extracted 350 / 792
  extracted 400 / 792
  extracted 450 / 792
  extracted 500 / 792
  extracted 550 / 792
  extracted 600 / 792
  extracted 650 / 792
  extracted 700 / 792
  extracted 750 / 792
total processed: 792
checkable claims: 759


In [37]:
# saving intermediate results 
claims_df.to_csv('../data/interim/claims_stage1.csv', index = False)
print('saved claims_stage1.csv:', len(claims_df), 'rows')

saved claims_stage1.csv: 792 rows


In [38]:
# checking the results to see whether the claims are good
for _, r in claims_df[claims_df['checkable']].iterrows():
    print('claim:', r['claim'])
    print('from:', r['title'][:90])
    print('abstract start:', r['abstract'][:200], '...')

claim: Ectopic expression of miR-140 inhibits the migratory and invasive capacities of colorectal cancer cells in vitro.
from: MicroRNA-140 Inhibits the Epithelial-Mesenchymal Transition and Metastasis in Colorectal C
abstract start: MicroRNA-140, a cartilage-specific microRNA, has recently been implicated in the cancer progression. However, the comprehensive role of miR-140 in the invasion and metastasis of colorectal cancer (CRC ...
claim: Microsurgical strabismus correction in children results in short operation time, less intraoperative blood loss, short hospital stay, high efficiency, and fewer complications.
from: Observation on the Curative Effect of Microsurgery in 154 Children with Strabismus and Ana
abstract start: Strabismus is a common ophthalmic disease in the process of child body development, in which the two eyes cannot gaze at the target at the same time, and the incidence of this disease of children is h ...
claim: A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can ac

In [39]:
# sanity check for control 
# loading full Retraction Watch database
# collecting every paper's PubMed ID into a set for quicker lookup

retraction_full = pd.read_csv('../data/raw/retraction_watch.csv', dtype = str, keep_default_na = False)

# collecting all useable original paper PubMed IDs
retracted_pmids = set(p.strip() for p in retraction_full['OriginalPaperPubMedID']
                      if p.strip().isdigit() and p.strip() != '0')

print('total retracted PMIDs to exclude:', len(retracted_pmids))

total retracted PMIDs to exclude: 31841


In [40]:
# prompting an LLM to search PubMed articles for relevant terms rather than a keyword match

QUERY_PROMPT = '''Turn this biomedical claim into a SHORT PubMed search query.
Output ONLY 2-4 key terms: the main entity (gene/drug/method) and the main condition or outcome.
Drop specific numbers, cell-line names and secondary details. Ensure the query is broad enough to find related papers.
Do not add anything else (no field tags, no quotes, no explanation, etc).

Claim: {claim}'''

# creating the function for the query 
def build_query(claim): 

    # asking the LLM for relevant PubMed search terms for a given claim
    try: 
        return call_llm(QUERY_PROMPT.format(claim = claim)).strip()

    # in case of errors 
    except Exception as e:
        print('error:', e)
        return ''

In [41]:
from Bio import Entrez

# function to search PubMed
def search_pubmed(query, exclude_pmid, max_results = 5):

    # searchihg PubMed for a query, returning potential PMIDs and excluding retractions and its associated paper
    try: 
        h = Entrez.esearch(db = 'pubmed', term = query, retmax = max_results * 3)
        rec = Entrez.read(h); h.close()
        hits = rec.get('IdList', [])

    # error handling
    except Exception as e:
        print('  search error:', e)
        return []

    # keeping only non-retracted papers
    clean = [pmid for pmid in hits
             if pmid not in retracted_pmids and str(pmid) != str(exclude_pmid)]

    return clean[:max_results]

In [42]:
# for each checkable claim, this cell builds the query, searches PubMed and finds the non-retraced papers

checkable = claims_df[claims_df['checkable']].copy()

retrieval_results = []
for i, (_, row) in enumerate(checkable.iterrows()):
    query = build_query(row['claim'])
    candidates = search_pubmed(query, exclude_pmid = row['pmid'])
    retrieval_results.append({
        'pmid': row['pmid'],
        'claim': row['claim'],
        'query': query,
        'candidate_pmids': candidates,
        'n_candidates': len(candidates),})

    # set to control PubMed retrieval rate
    time.sleep(0.34) 

    # progress bar
    if (i + 1) % 25 == 0:
        print(f'  retrieved candidates for {i + 1} / {len(checkable)}')

retrieval_df = pd.DataFrame(retrieval_results)

# calculations for viable papers
found = (retrieval_df['n_candidates'] > 0).sum()
print(f'claims with >= 1 candidate: {found} / {len(retrieval_df)}')
print(f'median candidates per claim: {int(retrieval_df['n_candidates'].median())}')

  retrieved candidates for 25 / 759
  retrieved candidates for 50 / 759
  retrieved candidates for 75 / 759
  retrieved candidates for 100 / 759
  retrieved candidates for 125 / 759
  retrieved candidates for 150 / 759
  retrieved candidates for 175 / 759
  retrieved candidates for 200 / 759
  retrieved candidates for 225 / 759
  retrieved candidates for 250 / 759
  retrieved candidates for 275 / 759
  retrieved candidates for 300 / 759
  retrieved candidates for 325 / 759
  retrieved candidates for 350 / 759
  retrieved candidates for 375 / 759
  retrieved candidates for 400 / 759
  search error: Remote end closed connection without response
  retrieved candidates for 425 / 759
  retrieved candidates for 450 / 759
  retrieved candidates for 475 / 759
  retrieved candidates for 500 / 759
  retrieved candidates for 525 / 759
  retrieved candidates for 550 / 759
  retrieved candidates for 575 / 759
  retrieved candidates for 600 / 759
  retrieved candidates for 625 / 759
  retrieved cand

In [43]:
# inspecting the query results
for _, r in retrieval_df.iterrows():
    print(f"[{r['n_candidates']} candidates] query: {r['query']}")
    print(f"  claim: {r['claim'][:90]}")
    print('-' * 90)

[5 candidates] query: miR-140 colorectal cancer migration invasion
  claim: Ectopic expression of miR-140 inhibits the migratory and invasive capacities of colorectal
------------------------------------------------------------------------------------------
[5 candidates] query: strabismus surgery children outcomes
  claim: Microsurgical strabismus correction in children results in short operation time, less intr
------------------------------------------------------------------------------------------
[5 candidates] query: MCA-PSV fetal anemia transfusions
  claim: A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately predict the recurrence of se
------------------------------------------------------------------------------------------
[0 candidates] query: sCLU recurrence-free survival
  claim: Elevated sCLU was significantly associated with recurrence-free survival in groups 1 and 3
------------------------------------------------------------------------------------------
[5

In [44]:
# getting the abstract text for every usable PMID; this is later used to score entailment against a claim
# retrieval_df has a list of usable PMIDs per claim

def fetch_candidates(pmid_list):

    # getting abstracts and returning the list of {pmid, abstract} dictionaries
    # skips those with no usable abstract
    out = []
    for pmid in pmid_list:
        abstract = fetch_abstract(pmid)
        time.sleep(0.34) 

        # keeping only usable abstracts
        if len(abstract.split()) > 20:   
            out.append({'pmid': pmid, 'abstract': abstract})
    return out

# progress tracker
candidates_col = []
for i, (_, row) in enumerate(retrieval_df.iterrows()):
    candidates_col.append(fetch_candidates(row['candidate_pmids']))
    if (i + 1) % 25 == 0:
        print(f'  fetched candidate abstracts for {i + 1} / {len(retrieval_df)}')

# fetching candidates for each claim
retrieval_df['candidates'] = candidates_col

# reporting how many claims still have >= 1 usable candidate after abstract fetching
retrieval_df['n_usable'] = retrieval_df['candidates'].apply(len)
print('claims with >=1 usable candidate abstract:', (retrieval_df['n_usable'] > 0).sum(), '/', len(retrieval_df))

  fetched candidate abstracts for 25 / 759
  fetched candidate abstracts for 50 / 759
  fetched candidate abstracts for 75 / 759
  fetched candidate abstracts for 100 / 759
  fetched candidate abstracts for 125 / 759
  fetched candidate abstracts for 150 / 759
  fetched candidate abstracts for 175 / 759
  fetched candidate abstracts for 200 / 759
  fetched candidate abstracts for 225 / 759
  fetched candidate abstracts for 250 / 759
  fetched candidate abstracts for 275 / 759
  fetched candidate abstracts for 300 / 759
  fetched candidate abstracts for 325 / 759
  fetched candidate abstracts for 350 / 759
  fetched candidate abstracts for 375 / 759
  fetched candidate abstracts for 400 / 759
  fetched candidate abstracts for 425 / 759
  fetched candidate abstracts for 450 / 759
  fetched candidate abstracts for 475 / 759
  fetched candidate abstracts for 500 / 759
  fetched candidate abstracts for 525 / 759
  fetched candidate abstracts for 550 / 759
  fetched candidate abstracts for 5

In [45]:
retrieval_df.to_pickle('../data/interim/retrieval_stage2.pkl')
print('saved retrieval_stage2.pkl:', len(retrieval_df), 'rows')

saved retrieval_stage2.pkl: 759 rows


In [46]:
# setting up the scientific NLI model 

# neural network packages
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# SciFact-fine-tuned DeBERTa: given a claim + evidence, classifies support / contradict / no-evidence
NLI_MODEL = 'MilosKosRad/TextualEntailment_DeBERTa_preprocessedSciFACT'

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)

# no training
model.eval()

# printing the label mappings
print('label mapping:', model.config.id2label)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

label mapping: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}


In [47]:
import torch

def raw_scores(premise, hypothesis):

    # returning the model's probability across the 3 labels 
    inputs = tokenizer(premise, hypothesis, return_tensors = 'pt', truncation = True, max_length = 512)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim = -1)[0]
    return probs.tolist()

# test 1 - case where the premise clearly supports the hypothesis
support_case = raw_scores(
    'Aspirin significantly reduces the risk of heart attack in adults.',
    'Aspirin lowers heart attack risk.'
)
print('support case:', [round(x, 3) for x in support_case])

# test 2 - premise clearly contradicts the hypothesis
contradict_case = raw_scores(
    'Aspirin significantly reduces the risk of heart attack in adults.',
    'Aspirin increases heart attack risk.'
)
print("conrtradiction case:", [round(x, 3) for x in contradict_case])

# test 3 - premise is unrelated to hypothesis
neutral_case = raw_scores(
    'The Eiffel Tower is located in Paris, France.',
    'Aspirin lowers heart attack risk.'
)
print('neutral case:', [round(x, 3) for x in neutral_case])

support case: [0.117, 0.857, 0.027]
conrtradiction case: [0.02, 0.008, 0.972]
neutral case: [0.995, 0.004, 0.001]


In [48]:
# scores how strongly a passage supports a claim using the NLI model
# premise = the evidence passage (abstract), hypothesis = the claim.

SUPPORT_INDEX = 1

# function t ocheck support
def support_score(passage, claim):

    # returning the probability that the modek's passage supports its claim (0 to 1)
    inputs = tokenizer(passage, claim, return_tensors = 'pt', truncation = True, max_length = 512)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim = -1)[0]
    return probs[SUPPORT_INDEX].item()

In [49]:
# building a flat file for exportation to Colab: one row per (claim, passage) that needs an NLI score

pmid_to_abstract = dict(zip(claims_df['pmid'], claims_df['abstract']))
gated = retrieval_df[retrieval_df['n_usable'] > 0].copy()

rows = []
for _, r in gated.iterrows():
    claim = r['claim']
    src_pmid = r['pmid']

    # getting the retracted paper's own abstract
    rows.append({'src_pmid': src_pmid, 'claim': claim,
                 'passage_pmid': src_pmid, 'role': 'retracted',
                 'passage': pmid_to_abstract.get(src_pmid, '')})
    
    # each candidate valid abstract
    for cand in r['candidates']:
        rows.append({'src_pmid': src_pmid, 'claim': claim,
                     'passage_pmid': cand['pmid'], 'role': 'candidate',
                     'passage': cand['abstract']})

# outputs
scoring_input = pd.DataFrame(rows)
scoring_input.to_csv('../data/interim/scoring_input.csv', index = False)
print('rows to score:', len(scoring_input))

rows to score: 2700


In [50]:
# updated code to load Colab GPU scores and rebuild gate_df 
scores = pd.read_csv('../data/interim/scores.csv', dtype = {'src_pmid': str, 'passage_pmid': str})

# retracted score per claim: one row where role == 'retracted'
retracted_scores = (scores[scores['role'] == 'retracted']
                    .set_index('src_pmid')['support_score'].to_dict())

# best valid candidate per claim: highest-scoring 'candidate' row
cand = scores[scores['role'] == 'candidate']
best_valid = cand.loc[cand.groupby('src_pmid')['support_score'].idxmax()]
best_valid_score = best_valid.set_index('src_pmid')['support_score'].to_dict()
best_valid_pmid  = best_valid.set_index('src_pmid')['passage_pmid'].to_dict()

# mapping the claim text back from claims_df
pmid_to_claim = dict(zip(claims_df['pmid'].astype(str), claims_df['claim']))

gate_rows = []
for src_pmid in retracted_scores:
    s_ret = retracted_scores[src_pmid]

    # -1 if no candidate scored
    s_val = best_valid_score.get(src_pmid, -1.0)
    gate_rows.append({
        'pmid': src_pmid,
        'claim': pmid_to_claim.get(src_pmid, ''),
        's_retracted': s_ret,
        'best_valid_pmid': best_valid_pmid.get(src_pmid),
        's_valid': s_val,
        'gap': abs(s_ret - s_val) if s_val >= 0 else 1.0,
    })

gate_df = pd.DataFrame(gate_rows)
print('claims scored:', len(gate_df))
print(gate_df[['s_retracted', 's_valid', 'gap']].describe())

claims scored: 536
       s_retracted     s_valid         gap
count   536.000000  536.000000  536.000000
mean      0.713432    0.885658    0.313252
std       0.342233    0.255692    0.333596
min       0.000949    0.000864    0.000002
25%       0.522594    0.961242    0.010263
50%       0.882704    0.998229    0.200168
75%       0.998112    0.999499    0.536838
max       0.999803    0.999832    0.998509


In [51]:
# filtering to keep a pair only if both abstracts support the claim (>= T) AND comparably (gap <= M)
T = 0.7
M = 0.2

# function to score
def passes_gate(row, t = T, m = M):
    return (row['s_retracted'] >= t) and (row['s_valid'] >= t) and (row['gap'] <= m)

gate_df['passes'] = gate_df.apply(passes_gate, axis = 1)

n_pass = gate_df['passes'].sum()
print(f'pairs passing gate (T={T}, M={M}): {n_pass} / {len(gate_df)}')

pairs passing gate (T=0.7, M=0.2): 253 / 536


In [ ]:
# sensitivity check to set threshold
for t, m in [(0.7, 0.2), (0.65, 0.25), (0.6, 0.25), (0.6, 0.3), (0.5, 0.3)]:
    n = gate_df.apply(lambda r: (r['s_retracted'] >= t) and (r['s_valid'] >= t) and (r['gap'] <= m), axis=1).sum()
    print(f'T={t}, M={m}: {n} pairs')

T=0.7, M=0.2: 253 pairs
T=0.65, M=0.25: 268 pairs
T=0.6, M=0.25: 272 pairs
T=0.6, M=0.3: 292 pairs
T=0.5, M=0.3: 297 pairs


In [53]:
# freezing the benchmark and parametesr
T, M = 0.7, 0.2
gate_df['passes'] = gate_df.apply(
    lambda r: (r['s_retracted'] >= T) and (r['s_valid'] >= T) and (r['gap'] <= M), axis = 1)

benchmark = gate_df[gate_df['passes']].copy()

# attaching the retraction reason (tier) for later per-reason analysis
pmid_to_reason = dict(zip(claims_df['pmid'].astype(str), claims_df['reason']))
benchmark['reason'] = benchmark['pmid'].map(pmid_to_reason)

benchmark.to_csv('../data/final/benchmark_pairs.csv', index = False)
print('frozen benchmark:', len(benchmark), 'to data/final/benchmark_pairs.csv')
print(benchmark[['s_retracted', 's_valid', 'gap']].describe())

frozen benchmark: 253 to data/final/benchmark_pairs.csv
       s_retracted     s_valid         gap
count   253.000000  253.000000  253.000000
mean      0.969757    0.979838    0.035051
std       0.054106    0.047920    0.052378
min       0.716878    0.737032    0.000002
25%       0.969873    0.990428    0.000901
50%       0.996916    0.998755    0.009212
75%       0.999250    0.999610    0.048432
max       0.999803    0.999832    0.199757


In [ ]:
# hand-verification to confirm
# 1) the claim is a real checkable finding
# 2) the retracted abstract supports it
# 3) the valid candidate supports it
sample_check = benchmark.sample(min(15, len(benchmark)), random_state = 1)

# pulling abstracts back for reading
pmid_to_abstract = dict(zip(claims_df['pmid'].astype(str), claims_df['abstract']))

for _, r in sample_check.iterrows():
    print('claim:', r['claim'])
    print(f's_retracted = {r['s_retracted']:.3f}  s_valid = {r['s_valid']:.3f}  valid_pmid = {r['best_valid_pmid']}')
    print('retracted abstract:', pmid_to_abstract.get(r['pmid'], '')[:250], '...')

In [25]:
# setting up the date to keep a pair only if both the retracted and valid abstracts clearly support the claim

# both abstracts must score at leats 70%
T = 0.7   

# scores must be within this margin of each other
M = 0.2   

# defining the gate function
def passes_gate(row, t = T, m = M):
    return (row['s_retracted'] >= t) and (row['s_valid'] >= t) and (row['gap'] <= m)

gate_df['passes'] = gate_df.apply(passes_gate, axis = 1)

n_pass = gate_df['passes'].sum()
print(f'pairs passing gate (T = {T}, M = {M}): {n_pass} / {len(gate_df)}\n')


for _, r in gate_df[gate_df['passes']].iterrows():
    print(f'retracted = {r['s_retracted']:.3f} valid = {r['s_valid']:.3f} gap = {r['gap']:.3f} | {r['claim'][:65]}')

# getting retraction reason
for _, r in gate_df[~gate_df['passes']].iterrows():
    reasons = []
    if r['s_retracted'] < T: reasons.append(f'retracted<{T}')
    if r['s_valid'] < T:     reasons.append(f'valid<{T}')
    if r['gap'] > M:         reasons.append(f'gap>{M}')
    print(f'[{', '.join(reasons)}] retracted = {r['s_retracted']:.3f} valid = {r['s_valid']:.3f} | {r['claim'][:60]}')

pairs passing gate (T = 0.7, M = 0.2): 10 / 14

retracted = 0.996 valid = 0.996 gap = 0.000 | miR-140 inhibits the tumor formation and metastasis of colorectal
retracted = 0.879 valid = 0.998 gap = 0.120 | Microsurgical strabismus correction in children results in short 
retracted = 0.998 valid = 0.999 gap = 0.001 | A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately pre
retracted = 0.999 valid = 0.843 gap = 0.156 | Stratified nursing management significantly improves nursing qual
retracted = 0.854 valid = 0.997 gap = 0.143 | Ultrasound-guided high-intensity focused ultrasound improves the 
retracted = 0.987 valid = 0.999 gap = 0.011 | Administration of a retroviral vector containing ACE antisense ca
retracted = 0.988 valid = 1.000 gap = 0.012 | The ADC value of the measured patient's osteomyelitis area was si
retracted = 0.987 valid = 1.000 gap = 0.012 | ARPE19, ARPE19mel, and hfRPE cells failed to form a tight barrier
retracted = 0.999 valid = 0.997 gap = 0.002 | Drainage a